# 06 - Job Profile and Candidate-Job Matching

Notebook này làm 2 việc lớn:

1. Xử lý job thô để tạo **job profile**.
2. So khớp **candidate profile** với **job profile** để tạo kết quả matching.

Mầm non recommendation:

```text
candidate profile + job profile
        ↓
so kỹ năng, nhóm kỹ năng, nhóm chính
        ↓
job nào hợp candidate hơn thì điểm cao hơn
```

## 1. Import thư viện

In [1]:
import ast
import re
from pathlib import Path

import pandas as pd

## 2. Khai báo đường dẫn

Input chính:

- `05_candidate_profiles.xlsx`: hồ sơ ứng viên từ bước 05.
- `job_raw.xlsx`: dữ liệu job thô.
- `14_skill_mapping_clean.xlsx`: mapping skill sạch.
- `08_djinni_step4_final.xlsx`: alias/tên gọi khác.

Output:

- `06_job_profiles.xlsx`: hồ sơ job đã xử lý.
- `07_candidate_job_matching.xlsx`: kết quả matching candidate-job.

In [2]:
BASE_DIR = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline")
DATAXOMY_DIR = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy")

CANDIDATE_PROFILE_PATH = BASE_DIR / "data_outputs" / "step05_candidate_profile" / "05_candidate_profiles.xlsx"
JOB_INPUT_PATH = BASE_DIR / "data_inputs" / "raw_job" / "job_raw.xlsx"

SKILL_MAPPING_PATH = DATAXOMY_DIR / "ESCO_taxonomy" / "notebook_clean" / "14_skill_mapping_clean.xlsx"
DJINNI_PATH = DATAXOMY_DIR / "Djinni" / "notebook_clean" / "08_djinni_step4_final.xlsx"

OUTPUT_DIR = BASE_DIR / "data_outputs" / "step06_job_and_matching"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JOB_PROFILE_OUTPUT_PATH = OUTPUT_DIR / "06_job_profiles.xlsx"
MATCHING_OUTPUT_PATH = OUTPUT_DIR / "07_candidate_job_matching.xlsx"

print("Candidate profile:", CANDIDATE_PROFILE_PATH, CANDIDATE_PROFILE_PATH.exists())
print("Job input:", JOB_INPUT_PATH, JOB_INPUT_PATH.exists())
print("Skill mapping:", SKILL_MAPPING_PATH, SKILL_MAPPING_PATH.exists())
print("Djinni:", DJINNI_PATH, DJINNI_PATH.exists())
print("Job profile output:", JOB_PROFILE_OUTPUT_PATH)
print("Matching output:", MATCHING_OUTPUT_PATH)

Candidate profile: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step05_candidate_profile/05_candidate_profiles.xlsx True
Job input: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_inputs/raw_job/job_raw.xlsx True
Skill mapping: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy/ESCO_taxonomy/notebook_clean/14_skill_mapping_clean.xlsx True
Djinni: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy/Djinni/notebook_clean/08_djinni_step4_final.xlsx True
Job profile output: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step06_job_and_matching/06_job_profiles.xlsx
Matching output: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/reco

## 3. Đọc dữ liệu

Đọc 4 bảng:

- candidate profile.
- job raw.
- skill mapping.
- alias Djinni.

In [3]:
candidate_df = pd.read_excel(CANDIDATE_PROFILE_PATH, engine="openpyxl")
job_df = pd.read_excel(JOB_INPUT_PATH, engine="openpyxl")
skill_df = pd.read_excel(SKILL_MAPPING_PATH, engine="openpyxl")
djinni_df = pd.read_excel(DJINNI_PATH, engine="openpyxl")

print("candidate_df:", candidate_df.shape)
print("job_df:", job_df.shape)
print("skill_df:", skill_df.shape)
print("djinni_df:", djinni_df.shape)

display(candidate_df.head(2))
display(job_df.head(2))

candidate_df: (20, 13)
job_df: (80, 3)
skill_df: (1171, 10)
djinni_df: (1171, 28)


,candidate_id,profile_text_clean,skills_raw_detected,mapped_skills,skill_groups,skill_subgroups,n_raw_skills,n_mapped_skills,dominant_group,section_summary,section_experience,section_projects,section_other
0,C001,frontend developer experienced in reactjs vuej...,"['javascript', 'implement frontend website des...","['JavaScript', 'CSS']",['Software Development'],['Web Development'],3,2,Software Development,NaN,NaN,NaN,frontend developer experienced in reactjs vuej...
1,C002,backend engineer experienced in java spring bo...,"['perform software unit testing', 'mysql']","['perform software unit testing', 'MySQL']",['Software Development'],"['Software Testing', 'Database Management']",2,2,Software Development,NaN,NaN,NaN,backend engineer experienced in java spring bo...


,job_id,job_title,job_text
0,J001,Angular Frontend Engineer,"Frontend Engineer required. Skills: Angular, R..."
1,J002,Next.js Web Developer,Frontend Web Developer required. Skills: Next....


## 4. Kiểm tra cột bắt buộc

Candidate phải có:

- `mapped_skills`
- `skill_groups`
- `dominant_group`

Job phải có:

- `job_id`
- `job_text`

In [4]:
required_candidate_cols = [
    "candidate_id",
    "mapped_skills",
    "skill_groups",
    "dominant_group",
    "profile_text_clean",
]

required_job_cols = ["job_id", "job_text"]


def check_required_cols(df, required_cols, df_name):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} thiếu cột: {missing}")


check_required_cols(candidate_df, required_candidate_cols, "candidate_df")
check_required_cols(job_df, required_job_cols, "job_df")

print("Input đủ cột cần thiết.")

Input đủ cột cần thiết.


## 5. Hàm tiện ích

Các hàm này dùng lại nhiều lần:

- chuẩn hóa text.
- parse list từ Excel.
- loại trùng nhưng giữ thứ tự.

In [5]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_skill_phrase(text):
    text = normalize_text(text)
    text = re.sub(r"[^\w\s\+\#\.\-/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_list(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    value = str(value).strip()
    if not value:
        return []

    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

    parts = re.split(r"[;,|]", value)
    return [p.strip() for p in parts if p.strip()]


def unique_preserve_order(values):
    seen = set()
    result = []

    for value in values:
        if pd.isna(value):
            continue

        value = str(value).strip()
        if not value:
            continue

        if value not in seen:
            seen.add(value)
            result.append(value)

    return result

## 6. Tìm cột skill trong file 14

Notebook tự nhận diện cột nào là tên skill, cột nào là nhóm skill.

Mục tiêu: tạo được bảng tra cứu skill → group.

In [6]:
possible_skill_name_cols = [
    "ten_sach", "skill_clean", "preferred_label", "skill_name", "skill", "ten_goc"
]

possible_group_cols = [
    "mapped_taxonomy_group",
    "taxonomy_group",
    "nhom_lon",
    "skill_group",
    "group",
]

possible_skill_subgroup_cols = [
    "mapped_taxonomy_subgroup",
    "taxonomy_subgroup",
    "skill_subgroup",
    "nhom_nho",
]

skill_name_col = next((c for c in possible_skill_name_cols if c in skill_df.columns), None)
group_col = next((c for c in possible_group_cols if c in skill_df.columns), None)
skill_subgroup_col = next((c for c in possible_skill_subgroup_cols if c in skill_df.columns), None)

print("skill_name_col =", skill_name_col)
print("group_col =", group_col)
print("skill_subgroup_col =", skill_subgroup_col)

if skill_name_col is None:
    raise ValueError("Không tìm thấy cột skill chuẩn trong file skill mapping.")

skill_name_col = skill_name
group_col = mapped_taxonomy_group
skill_subgroup_col = mapped_taxonomy_subgroup


## 7. Tạo `taxonomy_lookup`

`taxonomy_lookup` là sổ tra skill chuẩn.

```text
skill chuẩn → mapped_skill + group + skill_subgroup
```

Nó dùng để biết job skill thuộc nhóm nào.

In [7]:
taxonomy_lookup = {}

for _, row in skill_df.iterrows():
    canonical = normalize_skill_phrase(row.get(skill_name_col, ""))
    if canonical == "":
        continue

    taxonomy_lookup[canonical] = {
        "mapped_skill": row.get(skill_name_col, ""),
        "group": row.get(group_col, "") if group_col else "",
        "skill_subgroup": row.get(skill_subgroup_col, "") if skill_subgroup_col else "",
    }

print("Số skill trong taxonomy_lookup:", len(taxonomy_lookup))
list(taxonomy_lookup.items())[:5]

Số skill trong taxonomy_lookup: 1171


[('python computer programming',
  {'mapped_skill': 'Python (computer programming)',
   'group': 'Data & AI',
   'skill_subgroup': 'AI / Machine Learning'}),
 ('computer vision',
  {'mapped_skill': 'computer vision',
   'group': 'Data & AI',
   'skill_subgroup': 'AI / Machine Learning'}),
 ('deep learning',
  {'mapped_skill': 'deep learning',
   'group': 'Data & AI',
   'skill_subgroup': 'AI / Machine Learning'}),
 ('machine learning',
  {'mapped_skill': 'machine learning',
   'group': 'Data & AI',
   'skill_subgroup': 'AI / Machine Learning'}),
 ('utilise machine learning',
  {'mapped_skill': 'utilise machine learning',
   'group': 'Data & AI',
   'skill_subgroup': 'AI / Machine Learning'})]

## 8. Tạo `alias_to_canonical`

Job text có thể viết tên skill theo kiểu thị trường.

Ví dụ:

```text
reactjs → react
postgres → postgresql
js → javascript
```

Bước này tạo từ điển alias để bắt skill trong job tốt hơn.

In [8]:
possible_djinni_canonical_cols = [
    "ten_sach", "skill_clean", "skill_name", "ten_goc"
]

djinni_canonical_col = next(
    (c for c in possible_djinni_canonical_cols if c in djinni_df.columns),
    None,
)

possible_alias_cols = [
    "ten_ngoai_thi_truong", "cac_ten_gan_giong", "ten_goc"
]

djinni_alias_cols = [
    c for c in possible_alias_cols
    if c in djinni_df.columns
]

print("djinni_canonical_col =", djinni_canonical_col)
print("djinni_alias_cols =", djinni_alias_cols)


def parse_alias_value(value):
    if pd.isna(value):
        return []

    value = str(value).strip()
    if not value:
        return []

    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

    parts = re.split(r"[;,|]", value)
    return [p.strip() for p in parts if p.strip()]


alias_to_canonical = {}

if djinni_canonical_col is not None:
    for _, row in djinni_df.iterrows():
        canonical_norm = normalize_skill_phrase(row.get(djinni_canonical_col, ""))

        if not canonical_norm:
            continue

        alias_to_canonical[canonical_norm] = canonical_norm

        for alias_col in djinni_alias_cols:
            raw_value = row.get(alias_col, "")
            alias_list = parse_alias_value(raw_value)

            if not alias_list and pd.notna(raw_value):
                alias_list = [str(raw_value).strip()]

            for alias in alias_list:
                alias_norm = normalize_skill_phrase(alias)
                if alias_norm:
                    alias_to_canonical[alias_norm] = canonical_norm

print("Số alias trong Djinni:", len(alias_to_canonical))
list(alias_to_canonical.items())[:10]

djinni_canonical_col = ten_sach
djinni_alias_cols = ['ten_ngoai_thi_truong', 'cac_ten_gan_giong', 'ten_goc']
Số alias trong Djinni: 1368


[('python computer programming', 'python computer programming'),
 ('computer vision', 'computer vision'),
 ('deep learning', 'deep learning'),
 ('machine learning', 'machine learning'),
 ('ml', 'machine learning'),
 ('ai', 'machine learning'),
 ('model training', 'machine learning'),
 ('utilise machine learning', 'utilise machine learning'),
 ('frostbite digital game creation systems',
  'frostbite digital game creation systems'),
 ('ict accessibility standards', 'ict accessibility standards')]

## 9. Làm sạch job text

Job raw cũng là text bẩn.

Bước này tạo:

- `job_text_raw`: text gốc.
- `job_text_clean`: text đã chuẩn hóa để extract skill.

Nếu không có cột `job_title`, notebook tạo cột rỗng để tránh lỗi.

In [9]:
job_df = job_df.copy()

job_df["job_text_raw"] = job_df["job_text"].astype(str)
job_df["job_text_clean"] = job_df["job_text_raw"].apply(normalize_skill_phrase)

if "job_title" not in job_df.columns:
    job_df["job_title"] = ""

display(job_df[["job_id", "job_title", "job_text_clean"]].head())

,job_id,job_title,job_text_clean
0,J001,Angular Frontend Engineer,frontend engineer required. skills angular rxj...
1,J002,Next.js Web Developer,frontend web developer required. skills next.j...
2,J003,Frontend UI Engineer,ui engineer required. skills angular web compo...
3,J004,Vue Frontend Developer,vue developer required. skills vue.js nuxt.js ...
4,J005,React Frontend Developer,react frontend developer required. skills reac...


## 10. Tạo bộ phrase để tìm skill trong job

Bước này gom:

```text
skill chuẩn từ taxonomy_lookup
+ alias từ alias_to_canonical
= skill_lookup
```

Sau đó sort phrase dài trước để tránh match nhầm.

Ví dụ nên dò `machine learning` trước `learning`.

In [10]:
skill_lookup = {}

for skill in taxonomy_lookup.keys():
    skill_lookup[skill] = skill

for alias, canonical in alias_to_canonical.items():
    skill_lookup[alias] = canonical

skill_phrases_sorted = sorted(
    skill_lookup.keys(),
    key=lambda x: len(x),
    reverse=True,
)

print("Tổng phrase dùng để extract job skills:", len(skill_phrases_sorted))
print(skill_phrases_sorted[:30])

Tổng phrase dùng để extract job skills: 1368
['apply research ethics and scientific integrity principles in research activities', 'promote the participation of citizens in scientific and research activities', 'engage local communities in the management of natural protected areas', 'interact professionally in research and professional environments', 'communicate commercial and technical issues in foreign languages', 'draft scientific or academic papers and technical documentation', 'improve customer traveling experiences with augmented reality', 'develop professional network with researchers and scientists', 'manage findable accessible interoperable and reusable data', 'tourist resources of a destination for further development', 'draw sketches to develop textile articles using softwares', 'manage distribution of destination promotional materials', 'assemble health and safety resources in cultural venues', 'integrate marketing strategies with the global strategy', 'organise participatio

## 11. Extract skill từ job text

Bước này giống file 03 nhưng áp dụng cho job.

```text
job_text_clean
→ quét bằng skill_lookup
→ job_skills_detected
```

In [11]:
def extract_skills_from_text(text, skill_phrases, lookup_dict):
    text_norm = normalize_skill_phrase(text)
    matched = []

    if not text_norm:
        return []

    for phrase in skill_phrases:
        pattern = r"(?<!\w)" + re.escape(phrase) + r"(?!\w)"
        if re.search(pattern, text_norm):
            matched.append(lookup_dict[phrase])

    seen = set()
    result = []
    for skill in matched:
        if skill not in seen:
            seen.add(skill)
            result.append(skill)

    return result


job_df["job_skills_detected"] = job_df["job_text_clean"].apply(
    lambda text: extract_skills_from_text(text, skill_phrases_sorted, skill_lookup)
)

job_df["n_job_skills_detected"] = job_df["job_skills_detected"].apply(len)

display(job_df[[
    "job_id",
    "job_title",
    "job_skills_detected",
    "n_job_skills_detected",
]].head())

,job_id,job_title,job_skills_detected,n_job_skills_detected
0,J001,Angular Frontend Engineer,"[typescript, implement frontend website design...",3
1,J002,Next.js Web Developer,"[typescript, implement frontend website design]",2
2,J003,Frontend UI Engineer,"[typescript, implement frontend website design...",3
3,J004,Vue Frontend Developer,[typescript],1
4,J005,React Frontend Developer,"[typescript, implement frontend website design...",3


## 12. Map skill của job vào taxonomy

Sau khi tìm được skill trong job, bước này tra `taxonomy_lookup` để biết:

- skill chuẩn là gì.
- skill thuộc group nào.
- skill thuộc subgroup nào.

Kết quả là metadata của job.

In [12]:
def map_job_skill_list(skill_list):
    mapped_skills = []
    skill_groups = []
    skill_subgroups = []

    for skill in skill_list:
        skill_norm = normalize_skill_phrase(skill)

        if skill_norm in taxonomy_lookup:
            meta = taxonomy_lookup[skill_norm]

            mapped_skills.append(meta["mapped_skill"])

            if meta["group"]:
                skill_groups.append(meta["group"])

            if meta["skill_subgroup"]:
                skill_subgroups.append(meta["skill_subgroup"])
        else:
            mapped_skills.append(skill)

    return {
        "mapped_skills": unique_preserve_order(mapped_skills),
        "skill_groups": unique_preserve_order(skill_groups),
        "skill_subgroups": unique_preserve_order(skill_subgroups),
    }


job_meta = job_df["job_skills_detected"].apply(map_job_skill_list)
job_meta_df = pd.DataFrame(job_meta.tolist())

job_profile_df = pd.concat([job_df, job_meta_df], axis=1)
job_profile_df["n_mapped_skills"] = job_profile_df["mapped_skills"].apply(len)

display(job_profile_df[[
    "job_id",
    "job_title",
    "mapped_skills",
    "skill_groups",
    "skill_subgroups",
]].head())

,job_id,job_title,mapped_skills,skill_groups,skill_subgroups
0,J001,Angular Frontend Engineer,"[TypeScript, implement frontend website design...",[Software Development],[Frontend]
1,J002,Next.js Web Developer,"[TypeScript, implement frontend website design]",[Software Development],[Frontend]
2,J003,Frontend UI Engineer,"[TypeScript, implement frontend website design...",[Software Development],[Frontend]
3,J004,Vue Frontend Developer,[TypeScript],[Software Development],[Frontend]
4,J005,React Frontend Developer,"[TypeScript, implement frontend website design...",[Software Development],[Frontend]


## 13. Tính dominant group của job

Ở bản đơn giản này, dominant group của job lấy group đầu tiên trong danh sách `skill_groups`.

Ý nghĩa:

```text
job này nghiêng về nhóm kỹ năng/ngành nào nhất
```

In [13]:
def get_dominant_group(group_list):
    if not isinstance(group_list, list) or len(group_list) == 0:
        return ""

    return group_list[0]


job_profile_df["dominant_group"] = job_profile_df["skill_groups"].apply(get_dominant_group)

## 14. Xuất job profile

Output:

```text
06_job_profiles.xlsx
```

File này là hồ sơ job đã xử lý.

In [14]:
job_profile_output_df = job_profile_df[[
    "job_id",
    "job_title",
    "job_text_clean",
    "job_skills_detected",
    "mapped_skills",
    "skill_groups",
    "skill_subgroups",
    "n_job_skills_detected",
    "n_mapped_skills",
    "dominant_group",
]].copy()

job_profile_output_df.to_excel(JOB_PROFILE_OUTPUT_PATH, index=False)

print("Đã lưu job profiles:", JOB_PROFILE_OUTPUT_PATH)
display(job_profile_output_df.head())

Đã lưu job profiles: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step06_job_and_matching/06_job_profiles.xlsx


,job_id,job_title,job_text_clean,job_skills_detected,mapped_skills,skill_groups,skill_subgroups,n_job_skills_detected,n_mapped_skills,dominant_group
0,J001,Angular Frontend Engineer,frontend engineer required. skills angular rxj...,"[typescript, implement frontend website design...","[TypeScript, implement frontend website design...",[Software Development],[Frontend],3,3,Software Development
1,J002,Next.js Web Developer,frontend web developer required. skills next.j...,"[typescript, implement frontend website design]","[TypeScript, implement frontend website design]",[Software Development],[Frontend],2,2,Software Development
2,J003,Frontend UI Engineer,ui engineer required. skills angular web compo...,"[typescript, implement frontend website design...","[TypeScript, implement frontend website design...",[Software Development],[Frontend],3,3,Software Development
3,J004,Vue Frontend Developer,vue developer required. skills vue.js nuxt.js ...,[typescript],[TypeScript],[Software Development],[Frontend],1,1,Software Development
4,J005,React Frontend Developer,react frontend developer required. skills reac...,"[typescript, implement frontend website design...","[TypeScript, implement frontend website design...",[Software Development],[Frontend],3,3,Software Development


## 15. Chuẩn hóa candidate profile

Các cột list trong Excel có thể bị lưu thành chuỗi.

Bước này parse lại:

- `mapped_skills`
- `skill_groups`
- `skill_subgroups`

In [15]:
candidate_df = candidate_df.copy()

candidate_df["mapped_skills"] = candidate_df["mapped_skills"].apply(parse_list)
candidate_df["skill_groups"] = candidate_df["skill_groups"].apply(parse_list)
candidate_df["skill_subgroups"] = candidate_df["skill_subgroups"].apply(parse_list)

display(candidate_df[[
    "candidate_id",
    "mapped_skills",
    "skill_groups",
    "dominant_group",
]].head())

,candidate_id,mapped_skills,skill_groups,dominant_group
0,C001,"[JavaScript, CSS]",[Software Development],Software Development
1,C002,"[perform software unit testing, MySQL]",[Software Development],Software Development
2,C003,"[perform data analysis, SQL]",[Data & Databases],Data & Databases
3,C004,"[web services, DevOps]",[IT Infrastructure & Operations],IT Infrastructure & Operations
4,C005,[],[],NaN


07_match_candidate_job_clean_explained.ipynb

## 16. Hàm tính điểm matching

Có 3 loại điểm:

### 1. `skill_overlap_score`

Ứng viên có bao nhiêu skill mà job cần.

```text
candidate skill ∩ job skill / job skill
```

### 2. `group_similarity_score`

Nhóm skill của candidate và job giống nhau bao nhiêu.

Dùng Jaccard similarity:

```text
giao / hợp
```

### 3. `dominant_group_score`

Nhóm chính của candidate có trùng nhóm chính của job không.

- Trùng: 1.0
- Không trùng: 0.0

In [16]:
def jaccard_similarity(list_a, list_b): # Tính độ tương đồng nhóm skill giữa 2 list (bỏ qua khoảng trắng, chữ hoa/thường)
    set_a = set([
        str(x).strip().lower()
        for x in list_a
        if str(x).strip()
    ])

    set_b = set([
        str(x).strip().lower()
        for x in list_b
        if str(x).strip()
    ])

    if not set_a and not set_b:
        return 0.0

    if not set_a or not set_b:
        return 0.0

    return len(set_a & set_b) / len(set_a | set_b)


def overlap_ratio(candidate_skills, job_skills): # Tính tỉ lệ overlap giữa 2 list skill (bỏ qua khoảng trắng, chữ hoa/thường)
    cand_set = set([
        str(x).strip().lower()
        for x in candidate_skills
        if str(x).strip()
    ])

    job_set = set([
        str(x).strip().lower()
        for x in job_skills
        if str(x).strip()
    ])

    if not job_set:
        return 0.0

    return len(cand_set & job_set) / len(job_set)


def dominant_group_match(candidate_group, job_group): # Tính độ tương đồng giữa 2 nhóm skill (bỏ qua khoảng trắng, chữ hoa/thường)
    if not candidate_group or not job_group:
        return 0.0

    return (
        1.0
        if str(candidate_group).strip().lower() == str(job_group).strip().lower()
        else 0.0
    )

## 17. Tính matching cho từng cặp candidate-job

Với mỗi candidate và mỗi job, tính:

```text
final_score =
    0.60 * skill_overlap_score
  + 0.25 * group_similarity_score
  + 0.15 * dominant_group_score
```

Ý nghĩa:

- Skill trùng là quan trọng nhất.
- Nhóm kỹ năng giống nhau cũng quan trọng.
- Nhóm chính trùng nhau là tín hiệu bổ sung.

In [17]:
matching_rows = []

for _, cand in candidate_df.iterrows():
    for _, job in job_profile_output_df.iterrows():
        skill_overlap_score = overlap_ratio(
            cand["mapped_skills"],
            job["mapped_skills"],
        )

        group_similarity_score = jaccard_similarity(
            cand["skill_groups"],
            job["skill_groups"],
        )

        dominant_group_score = dominant_group_match(
            cand["dominant_group"],
            job["dominant_group"],
        )

        final_score = (
            0.65 * skill_overlap_score
            + 0.25 * group_similarity_score
            + 0.10 * dominant_group_score
        )

        matching_rows.append({
            "candidate_id": cand["candidate_id"],
            "job_id": job["job_id"],
            "job_title": job["job_title"],
            "candidate_mapped_skills": cand["mapped_skills"],
            "job_mapped_skills": job["mapped_skills"],
            "candidate_groups": cand["skill_groups"],
            "job_groups": job["skill_groups"],
            "candidate_dominant_group": cand["dominant_group"],
            "job_dominant_group": job["dominant_group"],
            "skill_overlap_score": round(skill_overlap_score, 4),
            "group_similarity_score": round(group_similarity_score, 4),
            "dominant_group_score": round(dominant_group_score, 4),
            "final_score": round(final_score, 4),
        })

matching_df = pd.DataFrame(matching_rows)

print("matching_df shape:", matching_df.shape)
display(matching_df.head(10))

matching_df shape: (1600, 13)


,candidate_id,job_id,job_title,candidate_mapped_skills,job_mapped_skills,candidate_groups,job_groups,candidate_dominant_group,job_dominant_group,skill_overlap_score,group_similarity_score,dominant_group_score,final_score
0,C001,J001,Angular Frontend Engineer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
1,C001,J002,Next.js Web Developer,"[JavaScript, CSS]","[TypeScript, implement frontend website design]",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
2,C001,J003,Frontend UI Engineer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
3,C001,J004,Vue Frontend Developer,"[JavaScript, CSS]",[TypeScript],[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
4,C001,J005,React Frontend Developer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.3333,1.0,1.0,0.5667
5,C001,J006,Svelte Frontend Developer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.3333,1.0,1.0,0.5667
6,C001,J007,Web Component Engineer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.3333,1.0,1.0,0.5667
7,C001,J008,Frontend Platform Engineer,"[JavaScript, CSS]","[TypeScript, implement frontend website design]",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
8,C001,J009,UI Dashboard Developer,"[JavaScript, CSS]","[TypeScript, implement frontend website design]",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
9,C001,J010,Design System Frontend Engineer,"[JavaScript, CSS]","[TypeScript, implement frontend website design]",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500


## 18. Sắp xếp job theo điểm

Với mỗi candidate, job nào có `final_score` cao hơn sẽ đứng trước.

In [18]:
matching_df = matching_df.sort_values(
    by=["candidate_id", "final_score"],
    ascending=[True, False],
).reset_index(drop=True)

display(matching_df.head(20))

,candidate_id,job_id,job_title,candidate_mapped_skills,job_mapped_skills,candidate_groups,job_groups,candidate_dominant_group,job_dominant_group,skill_overlap_score,group_similarity_score,dominant_group_score,final_score
0,C001,J042,Automation Test Engineer,"[JavaScript, CSS]",[JavaScript],[Software Development],[Software Development],Software Development,Software Development,1.0000,1.0,1.0,1.0000
1,C001,J048,Test Automation Developer,"[JavaScript, CSS]","[develop automated software tests, JavaScript]",[Software Development],[Software Development],Software Development,Software Development,0.5000,1.0,1.0,0.6750
2,C001,J005,React Frontend Developer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.3333,1.0,1.0,0.5667
3,C001,J006,Svelte Frontend Developer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.3333,1.0,1.0,0.5667
4,C001,J007,Web Component Engineer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.3333,1.0,1.0,0.5667
5,C001,J001,Angular Frontend Engineer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
6,C001,J002,Next.js Web Developer,"[JavaScript, CSS]","[TypeScript, implement frontend website design]",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
7,C001,J003,Frontend UI Engineer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
8,C001,J004,Vue Frontend Developer,"[JavaScript, CSS]",[TypeScript],[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
9,C001,J008,Frontend Platform Engineer,"[JavaScript, CSS]","[TypeScript, implement frontend website design]",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500


## 19. Lấy top K job cho mỗi candidate

Mặc định lấy top 5 job phù hợp nhất cho từng candidate.

In [19]:
# ===== SELECT TOP-K MATCHING RESULTS PER CANDIDATE =====
# Thầy yêu cầu đánh giá @10 và @20, nên mỗi candidate cần ít nhất 20 job để rank.
# Đổi top_k từ 5 -> 20 để file matching không bị cắt còn 5 job/candidate.

top_k = 20

top_match_df = (
    matching_df
    .groupby("candidate_id", as_index=False, group_keys=False)
    .apply(lambda x: x.head(top_k))
    .reset_index(drop=True)
)

print("top_k:", top_k)
print("top_match_df shape:", top_match_df.shape)
print("Candidates:", top_match_df["candidate_id"].nunique())
print("Jobs:", top_match_df["job_id"].nunique())
print("Min jobs per candidate:", top_match_df.groupby("candidate_id")["job_id"].count().min())
print("Max jobs per candidate:", top_match_df.groupby("candidate_id")["job_id"].count().max())

display(top_match_df.head(30))


top_k: 20
top_match_df shape: (400, 13)
Candidates: 20
Jobs: 53
Min jobs per candidate: 20
Max jobs per candidate: 20


/var/folders/17/pg_l_9551j17gx808_0c4q3w0000gn/T/ipykernel_18547/3383171865.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matching_df


,candidate_id,job_id,job_title,candidate_mapped_skills,job_mapped_skills,candidate_groups,job_groups,candidate_dominant_group,job_dominant_group,skill_overlap_score,group_similarity_score,dominant_group_score,final_score
0,C001,J042,Automation Test Engineer,"[JavaScript, CSS]",[JavaScript],[Software Development],[Software Development],Software Development,Software Development,1.0000,1.0,1.0,1.0000
1,C001,J048,Test Automation Developer,"[JavaScript, CSS]","[develop automated software tests, JavaScript]",[Software Development],[Software Development],Software Development,Software Development,0.5000,1.0,1.0,0.6750
2,C001,J005,React Frontend Developer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.3333,1.0,1.0,0.5667
3,C001,J006,Svelte Frontend Developer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.3333,1.0,1.0,0.5667
4,C001,J007,Web Component Engineer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.3333,1.0,1.0,0.5667
5,C001,J001,Angular Frontend Engineer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
6,C001,J002,Next.js Web Developer,"[JavaScript, CSS]","[TypeScript, implement frontend website design]",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
7,C001,J003,Frontend UI Engineer,"[JavaScript, CSS]","[TypeScript, implement frontend website design...",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
8,C001,J004,Vue Frontend Developer,"[JavaScript, CSS]",[TypeScript],[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500
9,C001,J008,Frontend Platform Engineer,"[JavaScript, CSS]","[TypeScript, implement frontend website design]",[Software Development],[Software Development],Software Development,Software Development,0.0000,1.0,1.0,0.3500


## 20. Tạo explanation đơn giản

Explanation là lý do match.

Nếu có skill overlap, group similarity hoặc dominant group match thì thêm vào phần giải thích.

In [20]:
def build_explanation(row):
    reasons = []

    if row["skill_overlap_score"] > 0:
        reasons.append(f"skill overlap={row['skill_overlap_score']}")

    if row["group_similarity_score"] > 0:
        reasons.append(f"group similarity={row['group_similarity_score']}")

    if row["dominant_group_score"] > 0:
        reasons.append("same dominant group")

    return "; ".join(reasons)


top_match_df["match_explanation"] = top_match_df.apply(build_explanation, axis=1)

display(top_match_df[[
    "candidate_id",
    "job_id",
    "job_title",
    "final_score",
    "match_explanation",
]].head(20))

,candidate_id,job_id,job_title,final_score,match_explanation
0,C001,J042,Automation Test Engineer,1.0000,skill overlap=1.0; group similarity=1.0; same ...
1,C001,J048,Test Automation Developer,0.6750,skill overlap=0.5; group similarity=1.0; same ...
2,C001,J005,React Frontend Developer,0.5667,skill overlap=0.3333; group similarity=1.0; sa...
3,C001,J006,Svelte Frontend Developer,0.5667,skill overlap=0.3333; group similarity=1.0; sa...
4,C001,J007,Web Component Engineer,0.5667,skill overlap=0.3333; group similarity=1.0; sa...
5,C001,J001,Angular Frontend Engineer,0.3500,group similarity=1.0; same dominant group
6,C001,J002,Next.js Web Developer,0.3500,group similarity=1.0; same dominant group
7,C001,J003,Frontend UI Engineer,0.3500,group similarity=1.0; same dominant group
8,C001,J004,Vue Frontend Developer,0.3500,group similarity=1.0; same dominant group
9,C001,J008,Frontend Platform Engineer,0.3500,group similarity=1.0; same dominant group


## 21. Lưu matching output

Xuất file:

```text
07_candidate_job_matching.xlsx
```

Đây là kết quả baseline matching của ProcessPipeline.

In [21]:
top_match_df.to_excel(MATCHING_OUTPUT_PATH, index=False)
print("Đã lưu matching output:", MATCHING_OUTPUT_PATH)

Đã lưu matching output: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step06_job_and_matching/07_candidate_job_matching.xlsx


## 22. Đọc lại output để kiểm tra

In [22]:
result = pd.read_excel(MATCHING_OUTPUT_PATH, engine="openpyxl")

print("Output shape:", result.shape)
display(result.head(20))

Output shape: (400, 14)


,candidate_id,job_id,job_title,candidate_mapped_skills,job_mapped_skills,candidate_groups,job_groups,candidate_dominant_group,job_dominant_group,skill_overlap_score,group_similarity_score,dominant_group_score,final_score,match_explanation
0,C001,J042,Automation Test Engineer,"['JavaScript', 'CSS']",['JavaScript'],['Software Development'],['Software Development'],Software Development,Software Development,1.0000,1.0,1,1.0000,skill overlap=1.0; group similarity=1.0; same ...
1,C001,J048,Test Automation Developer,"['JavaScript', 'CSS']","['develop automated software tests', 'JavaScri...",['Software Development'],['Software Development'],Software Development,Software Development,0.5000,1.0,1,0.6750,skill overlap=0.5; group similarity=1.0; same ...
2,C001,J005,React Frontend Developer,"['JavaScript', 'CSS']","['TypeScript', 'implement frontend website des...",['Software Development'],['Software Development'],Software Development,Software Development,0.3333,1.0,1,0.5667,skill overlap=0.3333; group similarity=1.0; sa...
3,C001,J006,Svelte Frontend Developer,"['JavaScript', 'CSS']","['TypeScript', 'implement frontend website des...",['Software Development'],['Software Development'],Software Development,Software Development,0.3333,1.0,1,0.5667,skill overlap=0.3333; group similarity=1.0; sa...
4,C001,J007,Web Component Engineer,"['JavaScript', 'CSS']","['TypeScript', 'implement frontend website des...",['Software Development'],['Software Development'],Software Development,Software Development,0.3333,1.0,1,0.5667,skill overlap=0.3333; group similarity=1.0; sa...
5,C001,J001,Angular Frontend Engineer,"['JavaScript', 'CSS']","['TypeScript', 'implement frontend website des...",['Software Development'],['Software Development'],Software Development,Software Development,0.0000,1.0,1,0.3500,group similarity=1.0; same dominant group
6,C001,J002,Next.js Web Developer,"['JavaScript', 'CSS']","['TypeScript', 'implement frontend website des...",['Software Development'],['Software Development'],Software Development,Software Development,0.0000,1.0,1,0.3500,group similarity=1.0; same dominant group
7,C001,J003,Frontend UI Engineer,"['JavaScript', 'CSS']","['TypeScript', 'implement frontend website des...",['Software Development'],['Software Development'],Software Development,Software Development,0.0000,1.0,1,0.3500,group similarity=1.0; same dominant group
8,C001,J004,Vue Frontend Developer,"['JavaScript', 'CSS']",['TypeScript'],['Software Development'],['Software Development'],Software Development,Software Development,0.0000,1.0,1,0.3500,group similarity=1.0; same dominant group
9,C001,J008,Frontend Platform Engineer,"['JavaScript', 'CSS']","['TypeScript', 'implement frontend website des...",['Software Development'],['Software Development'],Software Development,Software Development,0.0000,1.0,1,0.3500,group similarity=1.0; same dominant group


## Tóm tắt file 06

File 06 làm 2 việc:

```text
job_raw.xlsx
        ↓
extract skill + map taxonomy
        ↓
06_job_profiles.xlsx
```

Sau đó:

```text
05_candidate_profiles.xlsx
+ 06_job_profiles.xlsx
        ↓
tính skill_overlap_score
tính group_similarity_score
tính dominant_group_score
        ↓
07_candidate_job_matching.xlsx
```

Nói ngắn gọn:

> File 06 tạo hồ sơ job rồi tính điểm phù hợp giữa từng candidate và từng job.


File 06_job_profile_and_matching.ipynb dùng để tạo job profile và tính baseline matching. Đầu tiên, notebook đọc candidate profile từ file 05, đọc job raw, rồi dùng skill mapping và alias để trích xuất skill từ job text. Sau đó, các skill của job được map vào taxonomy để tạo job profile gồm mapped skills, skill groups và dominant group. Cuối cùng, hệ thống so khớp candidate profile với job profile bằng ba điểm: skill overlap, group similarity và dominant group score. Ba điểm này được kết hợp thành baseline final score, sau đó sắp xếp giảm dần để lấy top K job phù hợp cho từng ứng viên.

Chốt lại: bạn hiểu đúng rồi, chỉ sửa mỗi đoạn “tỉ lệ 1.0/0.5 dựa trên length text” thành dựa trên số skill trùng so với số skill job yêu cầu.